<a href="https://colab.research.google.com/github/DanisStepanov/image_processing/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%965_%D0%9A%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D1%8F_%D0%B8_%D0%BA%D0%BB%D0%B0%D1%81%D1%82%D0%B5%D1%80%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F_%D1%81%D0%BF%D1%83%D1%82%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2%D1%8B%D1%85_%D1%81%D0%BD%D0%B8%D0%BC%D0%BA%D0%BE%D0%B2_%D0%B2_GEE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №5. Сравнительный анализ контролируемой классификации и неконтролируемой кластеризации спутниковых снимков в Google Earth Engine: выявление ошибок пропуска и скрытых пространственных структур**

## **1. Цель работы**

Обучить в Google Earth Engine классификаторы земного покрова по разметке Dynamic World и честно оценить их точность, найти кластеризацией на снимке территории объекты, которых нет в легенде готовых карт, и экспериментально определить, когда кластеризация цветов действительно уменьшает размер изображений.

## **2. Задачи работы**

1.  **Подготовить** летний композит Sentinel-2 и карту Dynamic World для территории своего варианта, рассмотреть снимок в естественных цветах и в инфракрасной комбинации каналов и записать гипотезы о том, что на нём можно найти;
2.  **Обучить** несколько классификаторов GEE на стратифицированной выборке с метками Dynamic World, сравнить случайную и пространственную проверку точности, оценить точность по классам и важность признаков;
3.  **Построить** карту классификации территории и сравнить её с картой Dynamic World;
4.  **Кластеризовать** снимок алгоритмом K-средних на серверах GEE при нескольких значениях $K$, дать кластерам названия, найти малые и необычные группы пикселей и проверить гипотезы;
5.  **Провести** эксперименты по сжатию фотографий, карты и снимка кластеризацией цветов: размер файлов, время, качество, сравнение со встроенным методом квантизации;
6.  **Сформулировать** выводы с конкретными числами.

> **Примечание.** Работа опирается на две лекции:
> - [лекция «Экспорт растровых данных и применение методов машинного обучения в среде Google Earth Engine»](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2) — **Разделы 1–4** (композиты, экспорт, выборки, классификаторы);
> - лекция «Кластеризация спутниковых снимков в среде Google Earth Engine» — **Разделы 2–6** (K-средних, интерпретация кластеров, поиск необычного, сжатие изображений).
>
> Код из лекций можно использовать как образец, но задания этой работы отличаются от лекционных примеров: другая разметка, другая территория, другие проверки.

> 📝 **Как выполнять работу.** Выполняйте ячейки по порядку. В ячейки `# Ваш код` впишите свой код; если в ячейке уже есть заготовка, замените в ней `...`. Ответы на вопросы пишите в ячейки **Ответ:** — двойной щелчок по ячейке, текст ответа, затем Shift+Enter. В ответах приводите числа из своих результатов.

---

## **3. Выбор территории и подготовка данных в Google Earth Engine**

### **3.1. Настройка рабочей среды и аутентификация**

In [ ]:
%%capture
!pip install -U geemap geedim rasterio scikit-learn scikit-image

In [ ]:
# Импорт основных библиотек для работы
import os
import time

import ee                        # Google Earth Engine
import geemap                    # Интерактивные карты и экспорт растров из GEE
import numpy as np               # Работа с массивами
import pandas as pd              # Работа с таблицами
import rasterio                  # Чтение экспортированных GeoTIFF
import matplotlib.pyplot as plt  # Построение графиков
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image            # Сохранение изображений, в том числе палитровых PNG
from sklearn.cluster import KMeans
from skimage import data as skdata
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

%config InlineBackend.figure_format = 'jpeg'  # рисунки со снимками компактнее в JPEG

Перед началом работы с geemap и Google Earth Engine необходимо:

- **Зарегистрировать аккаунт Google Earth Engine**: Перейдите на [https://code.earthengine.google.com/register](https://code.earthengine.google.com/register) для создания аккаунта GEE. После регистрации нужно настроить проект Google Cloud и активировать API Earth Engine, следуя инструкциям [здесь](https://docs.google.com/document/d/1ZGSmrNm6_baqd8CHt33kIBWOlvkh-HLr46bODgJN1h0/edit?usp=sharing). GEE бесплатен для [некоммерческого и исследовательского использования](https://earthengine.google.com/noncommercial).

- **Проверить аутентификацию API**: Чтобы убедиться в правильности настройки аккаунта Earth Engine, запустите [этот тестовый блокнот](https://colab.research.google.com/github/gee-community/geemap/blob/master/docs/notebooks/geemap_colab.ipynb) в Google Colab.

In [ ]:
# Ваш код: ee.Authenticate() и ee.Initialize(project=...) с именем вашего проекта

### **3.2. Выбор территории**

Ниже перечислены территории, на которых есть объекты, плохо заметные на снимке в естественных цветах или отсутствующие в легенде готовых карт земного покрова: месторождения полезных ископаемых, нефтепромыслы, лесные гари, солепромыслы. Что именно и где искать, в таблице не сказано — это и предстоит выяснить. Номер варианта назначает преподаватель (если не назначен — номер в списке группы по модулю 7, нулю соответствует вариант 7).

#### **Таблица 1. Варианты территорий**

| **Вариант** | **Территория** | **Широта, долгота центра** | **Природная зона** |
|---|---|---|---|
| 1 | Олимпиадинское золоторудное месторождение, Красноярский край | 59.585, 92.990 | Южная тайга |
| 2 | Норильск: медно-никелевые и платиноидные руды, Красноярский край | 69.350, 88.200 | Лесотундра |
| 3 | Губкин, железные руды Курской магнитной аномалии, Белгородская область | 51.290, 37.700 | Лесостепь |
| 4 | Железногорск, железные руды Курской магнитной аномалии, Курская область | 52.360, 35.400 | Лесостепь |
| 5 | Самотлорское нефтяное месторождение, Ханты-Мансийский АО | 61.130, 76.750 | Средняя тайга, болота |
| 6 | Тайга в Забайкальском крае, район весенних лесных пожаров 2024 года | 54.665, 118.893 | Горная тайга |
| 7 | Озеро Баскунчак, Астраханская область | 48.200, 46.880 | Полупустыня |

Область интереса — квадрат 14×14 км вокруг центра. Все варианты анализируются по снимкам лета 2024 года.

In [ ]:
PLACES = {
    1: ('Олимпиадинское золоторудное месторождение', 59.585, 92.990),
    2: ('Норильск', 69.350, 88.200),
    3: ('Губкин, Лебединский ГОК', 51.290, 37.700),
    4: ('Железногорск, Михайловский ГОК', 52.360, 35.400),
    5: ('Самотлорское нефтяное месторождение', 61.130, 76.750),
    6: ('Тайга в Забайкальском крае', 54.665, 118.893),
    7: ('Озеро Баскунчак', 48.200, 46.880),
}
YEAR = 2024

In [ ]:
VARIANT = ...   # номер вашего варианта
# Ваш код: PLACE, LAT, LON = PLACES[VARIANT]; roi — квадрат 14×14 км: ee.Geometry.Point([LON, LAT]).buffer(7000).bounds()
# Ваш код: карта geemap с подложкой 'Esri.WorldImagery' и контуром roi

**Задание 3.2.** Рассмотрите территорию на карте-подложке. Ответьте до начала анализа.

**Вопрос:** Какие типы земного покрова вы видите на подложке? Какие объекты, связанные с названием территории (добыча, промыслы, пожары), вы ожидаете найти и где? Какие из них, по-вашему, не имеют своего класса в легенде Dynamic World (вода, деревья, трава, затопляемая растительность, сельхозкультуры, кустарники, застройка, голая земля, снег и лёд)?

**Ответ:** _______________

### **3.3. Летний композит Sentinel-2 и спектральные индексы**

> 📚 **Подсказка:** [лекция по GEE](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2), **разделы 2.2–2.3** (фильтрация, маска облаков по каналу SCL, медианный композит) и **3.3** (индексы).

**Задание 3.3.** Постройте медианный композит `composite` снимков Sentinel-2 уровня 2A (`COPERNICUS/S2_SR_HARMONIZED`) за 1 июня – 31 августа 2024 года:
- каналы `BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']`, значения отражения переведены в диапазон 0–1 (деление на 10000);
- снимки с облачностью `CLOUDY_PIXEL_PERCENTAGE` < 20%, облака и тени замаскированы по каналу SCL (классы 3, 8, 9, 10);
- к композиту добавлены индексы `NDVI` (B8, B4), `NDWI` (B3, B8) и `NDBI` (B11, B8);
- список всех признаков сохраните в переменную `FEATURES` — он понадобится для обучения классификаторов.

In [ ]:
# Ваш код

### **3.4. Карта Dynamic World**

Dynamic World — карта земного покрова с пикселем 10 м, которую Google строит нейросетью для каждого снимка Sentinel-2. В этой работе она служит **разметкой для обучения**: классификатор будет учиться повторять её классы по спектру пикселей.

**Задание 3.4.** Получите `dynamic_world` — моду канала `label` коллекции `GOOGLE/DYNAMICWORLD/V1` за тот же период (наиболее частый класс в каждом пикселе), обрежьте по `roi` и переименуйте канал в `'class'`.

In [ ]:
DW_NAMES = ['Вода', 'Деревья', 'Трава', 'Затопл. растительность', 'Культуры',
            'Кустарники', 'Застройка', 'Голая земля', 'Снег и лед']
DW_COLORS = ['#419bdf', '#397d49', '#88b053', '#7a87c6', '#e49635',
             '#dfc35a', '#c4281b', '#a59b8f', '#b39fe1']

In [ ]:
# Ваш код

### **3.5. Экспорт и просмотр растров**

Для анализа на локальной машине нужны шесть основных каналов композита, индексы NDVI и NDWI и карта Dynamic World. Растры экспортируются в проекции UTM с пикселем 10 м и загружаются плитками 256×256 пикселей. Карты классов сохраняются с типом `int16`: при типе `uint8` значение 0 используется как «нет данных» и совпало бы с классом «Вода». **Если экспорт идёт слишком долго, увеличьте `SCALE` до 20 м.**

In [ ]:
#@title Готовая функция экспорта
SCALE = 10  # размер пикселя, м

def utm_crs(region):
    """Код EPSG зоны UTM для центра области."""
    lon, lat = region.centroid(maxError=1).coordinates().getInfo()
    return f'EPSG:{(32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1}'

def export_georeferenced_raster(image, region, filename, scale=SCALE, dtype='float32', tile_size=256):
    """Экспорт в GeoTIFF в проекции UTM с загрузкой плитками."""
    geemap.download_ee_image(image=image, filename=filename, region=region, scale=scale,
                             crs=utm_crs(region), dtype=dtype, max_tile_dim=tile_size)
    return filename

def read_raster(filename):
    """Массив каналов растра (каналы, строки, столбцы)."""
    with rasterio.open(filename) as src:
        return src.read()

os.makedirs('data', exist_ok=True)

**Задание 3.5.** Экспортируйте каналы `LOCAL_BANDS` композита в `data/sentinel2.tif` (тип `float32`) и `dynamic_world` в `data/dynamic_world.tif` (тип `int16`). Загрузите их в массивы `bands` и `dw`, получите маску пикселей с данными `valid` (все каналы конечны и `dw >= 0`), размеры `H`, `W` и площадь пикселя в гектарах `PIXEL_AREA_HA`.

In [ ]:
LOCAL_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI']

In [ ]:
# Ваш код

Человеческий глаз видит только видимый диапазон, поэтому многие объекты на снимке в естественных цветах неразличимы. Комбинация **B12 – B8 – B4** (коротковолновый ИК, ближний ИК, красный) отображает здоровую растительность ярко-зелёной (высокое отражение в B8), открытый грунт, горные породы и гари — красно-бурыми и сиреневыми (высокое отражение в B12), воду — почти чёрной. Покажем снимок в обеих комбинациях и карту Dynamic World.

In [ ]:
#@title Готовая функция: снимок в естественных цветах и в комбинации B12-B8-B4
def stretch(x, high):
    """Линейное растяжение значений отражения в диапазон 0-1."""
    return np.clip(np.nan_to_num(x, nan=0, posinf=0, neginf=0) / high, 0, 1)

rgb = stretch(np.dstack([bands[2], bands[1], bands[0]]), 0.3) ** (1 / 1.4)   # B4, B3, B2
swir = np.dstack([stretch(bands[5], 0.4), stretch(bands[3], 0.45), stretch(bands[2], 0.25)])   # B12, B8, B4
dw_cmap = ListedColormap(DW_COLORS)
dw_legend = [Patch(color=col, label=name) for col, name in zip(DW_COLORS, DW_NAMES)]

fig, axes = plt.subplots(1, 3, figsize=(30, 10))
axes[0].imshow(rgb)
axes[0].set_title('Естественные цвета (B4, B3, B2)', fontsize=16)
axes[1].imshow(swir)
axes[1].set_title('Коротковолновый ИК (B12, B8, B4)', fontsize=16)
axes[2].imshow(np.ma.masked_less(dw, 0), cmap=dw_cmap, vmin=-0.5, vmax=8.5, interpolation='nearest')
axes[2].set_title('Dynamic World', fontsize=16)
axes[2].legend(handles=dw_legend, loc='lower left', fontsize=11)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

dw_areas = pd.Series(np.bincount(dw[valid], minlength=9) * PIXEL_AREA_HA, index=DW_NAMES, name='Площадь, га')
dw_areas[dw_areas > 0].sort_values(ascending=False).round(1)

**Задание 3.5 (продолжение).** Сравните три изображения. Запишите гипотезы: какие участки выглядят необычно на комбинации B12–B8–B4, хотя на снимке в естественных цветах не выделяются (или наоборот), и к какому классу Dynamic World их отнёс. Гипотезы проверите в разделе 5.

#### **Таблица 2. Гипотезы до анализа**

| № | Наблюдение (где, как выглядит в естественных цветах и в B12–B8–B4) | Класс Dynamic World | Гипотеза: что это за объект | Как проверить |
|---|---|---|---|---|
| 1 | ... | ... | ... | ... |
| 2 | ... | ... | ... | ... |
| 3 | ... | ... | ... | ... |

<details>
<summary><b>Пример формулировки</b></summary>

*Условный пример:* «В центре — светлое пятно неправильной формы около 2×3 км, в естественных цветах серо-бежевое, в B12–B8–B4 сиреневое, вокруг ярко-зелёная тайга. Dynamic World относит его к кустарникам и голой земле. Гипотеза: карьер и отвалы пустой породы. Проверка: отдельный кластер со спектром без растительности и высоким отражением в B11–B12».

</details>

**Ответ:** _______________

---

## **4. Классификация с учителем по разметке Dynamic World**

В лекции классификаторы обучались по карте ESA WorldCover. Здесь разметкой будет Dynamic World: классификатор учится по спектру пикселя предсказывать класс, который поставил бы Dynamic World. Это удобно — разметку не нужно создавать вручную, — но у такого подхода есть пределы: модель повторит и классы, и ошибки разметки. Их мы и увидим.

### **4.1. Стратифицированная обучающая выборка**

Классы на территории занимают очень разную площадь: например, тайга — десятки квадратных километров, застройка — несколько гектаров. Простая случайная выборка (`sample`) почти не содержит точек редких классов, и модель их не выучит. **Стратифицированная выборка** (`stratifiedSample`) берёт заданное число точек из каждого класса: если пикселей класса меньше, берутся все.

**Параметры метода `stratifiedSample`:**
- `numPoints` — число точек на класс
- `classBand` — канал с метками классов
- `region`, `scale` — область и размер пикселя выборки
- `seed` — начальное значение генератора для воспроизводимости
- `tileScale` — дробление вычислений на более мелкие плитки (помогает избежать ошибки нехватки памяти)

К каждой точке сразу добавим номер блока «шахматной доски» со стороной 1 км — он понадобится для пространственной проверки в разделе 4.2.

In [ ]:
# Номер блока шахматной доски 1×1 км: 0 — «чёрные» клетки, 1 — «белые»
BLOCK_SIZE = 1000
blocks = (ee.Image.pixelCoordinates(ee.Projection(utm_crs(roi)))
          .divide(BLOCK_SIZE).floor().reduce('sum').mod(2).toInt().rename('block'))

**Задание 4.1.** Получите выборку `samples`: по 400 точек каждого класса из изображения `composite.addBands(dynamic_world).addBands(blocks)` (`scale=SCALE`, `projection=utm_crs(roi)`, `seed=42`, `tileScale=4`). Проекция UTM обязательна: по умолчанию выборка берётся в географических координатах, где пиксель «10 м» по долготе в высоких широтах вдвое уже, и точки редких классов дублируются. Выведите число точек по классам (`aggregate_histogram('class')`) с названиями классов. Классы, у которых меньше 100 точек, исключите из выборки (`ee.Filter.inList`): по нескольким десяткам точек нельзя ни обучить, ни проверить модель. Список оставшихся кодов классов сохраните в `CLASSES`.

In [ ]:
# Ваш код

### **4.2. Случайная и пространственная проверка точности**

В лекции выборка делилась на обучающую и проверочную случайно (70/30). Но соседние пиксели снимка почти одинаковы: если один пиксель поля попал в обучение, а соседний — в проверку, модель «узнаёт» знакомое место, а не применяет выученное правило. Точность на такой проверке завышена. Честнее проверять модель на **других участках** территории: обучать на «чёрных» клетках шахматной доски, а проверять на «белых». Так оценивают, как модель поведёт себя на новом месте.

**Задание 4.2.** Создайте два разбиения выборки:
- случайное: `randomColumn('random', 42)`, обучение — `random < 0.7`, проверка — остальные;
- пространственное: обучение — `block == 0`, проверка — `block == 1`.

Для каждого разбиения обучите `ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42)` на признаках `FEATURES` и рассчитайте общую точность и каппу на проверочной части. Сохраните разбиения в словарь `splits = {'Случайная': (train, test), 'Пространственная': (train, test)}` и выведите таблицу результатов.

> 📚 **Подсказка:** [лекция по GEE](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2), **раздел 4.2** (разделение по `randomColumn`) и **4.3** (`train`, `classify`, `errorMatrix`).

In [ ]:
# Ваш код

**Вопрос:** На сколько различаются точность и каппа случайного леса при случайной и пространственной проверке? Почему? Какой оценке вы бы доверяли, если модель нужно применить к соседнему участку тайги или к другому месторождению?

**Ответ:** _______________

### **4.3. Сравнение классификаторов**

**Задание 4.3.** Выберите не менее трёх классификаторов из лекции (раздел 4.3) и сравните их **на пространственном разбиении**. Наивный байесовский классификатор не берите: он рассчитан на неотрицательные целые признаки (счётчики), а у нас дробные отражения и индексы. Сохраните обученные модели в словарь `models`, матрицы ошибок — в `matrices`, таблицу точности и каппы — в `metrics`, название лучшей модели по каппе — в `best_name`.

In [ ]:
classifiers = {
    'Random Forest': ee.Classifier.smileRandomForest(numberOfTrees=100, seed=42),
    # Ваш код: добавьте ещё не менее двух классификаторов
}
# Ваш код: обучение и проверка на splits['Пространственная'], словари models и matrices, таблица metrics, best_name

### **4.4. Точность по классам**

Общая точность скрывает, какие классы модель путает. Для каждого класса считают две точности:

- **точность производителя** (`producersAccuracy`, она же полнота) — какая доля пикселей класса в разметке распознана моделью правильно; низкое значение означает, что модель **пропускает** класс;
- **точность пользователя** (`consumersAccuracy`, она же точность в смысле precision) — какая доля пикселей, отнесённых моделью к классу, действительно относится к нему; низкое значение означает, что модель **приписывает** классу чужие пиксели.

*Условный пример:* если из 100 пикселей «Застройки» модель нашла 60, точность производителя 0,6; если при этом она назвала «Застройкой» 80 пикселей и 60 из них верны, точность пользователя 0,75.

**Задание 4.4.** Для лучшей модели постройте таблицу `per_class`: название класса, точность производителя и точность пользователя для классов из `CLASSES`. Матрица ошибок GEE индексируется кодами классов, поэтому строка `producersAccuracy()` — это список длиной «максимальный код + 1». Постройте также тепловую карту матрицы ошибок, нормированной по строкам (истинным классам).

In [ ]:
# Ваш код

**Вопрос:** Какие классы модель распознаёт лучше всего и хуже всего? С какими классами путается самый проблемный класс? Объясните, почему спектрально эти классы похожи.

**Ответ:** _______________

### **4.5. Важность признаков**

Случайный лес оценивает, насколько каждый признак помогает разделять классы. В GEE оценки возвращает метод `explain()` обученной модели (ключ `importance`).

**Задание 4.5.** Получите важность признаков модели `models['Random Forest']`, переведите её в доли от суммы и постройте горизонтальную столбчатую диаграмму, отсортированную по убыванию важности.

In [ ]:
# Ваш код

**Вопрос:** Какие три признака наиболее важны? Какую долю важности дают инфракрасные каналы (B8, B8A, B11, B12) вместе с индексами, использующими их? Что это говорит о пользе каналов за пределами видимого диапазона?

**Ответ:** _______________

### **4.6. Карта классификации и сравнение с Dynamic World**

Для построения итоговой карты обучим лучшую модель заново **на всей выборке** (обучающей и проверочной частях): проверка уже выполнена, и лишние точки сделают модель только точнее.

**Задание 4.6.** Обучите классификатор `best_name` на всей выборке `samples`, классифицируйте `composite.select(FEATURES)`, экспортируйте карту в `data/classification.tif` (тип `int16`) и загрузите её в массив `classification`. Рассчитайте:
- долю пикселей, в которых классификация совпадает с Dynamic World;
- таблицу `areas`: площадь каждого класса по Dynamic World и по своей карте (га);
- маску несовпадений и покажите её на снимке рядом с обеими картами.

In [ ]:
# Ваш код

**Вопрос:** Какова доля совпадения карт? Где сосредоточены несовпадения — на границах объектов, в каких-то конкретных типах местности? Площади каких классов различаются сильнее всего?

**Ответ:** _______________

**Вопрос:** Рассмотрите объекты, которые вы описали в гипотезах (таблица 2). К каким классам отнесли их Dynamic World и ваша модель? Может ли классификатор, обученный по разметке Dynamic World, выделить такие объекты как отдельный класс? Почему?

**Ответ:** _______________

### **4.7. Выводы по разделу 4**

Опишите, насколько точно модель воспроизводит Dynamic World при честной проверке, какие классы и признаки для неё важнее, и в чём главное ограничение обучения по готовой карте.

**Вывод:** _______________

---

## **5. Кластеризация в Google Earth Engine: поиск того, чего нет в легенде**

Классификатор умеет только то, чему его научили: каждому пикселю он присваивает один из классов разметки. Кластеризация не знает никаких классов и группирует пиксели по сходству спектра. Если на территории есть объекты, спектрально непохожие на всё остальное, они могут образовать собственный кластер — даже если в легенде для них нет класса и на снимке в естественных цветах они почти не видны.

### **5.1. Стандартизация признаков в GEE**

Кластеризовать будем по восьми признакам `LOCAL_BANDS`: шести каналам отражения и индексам NDVI и NDWI. Отражение меняется в пределах 0–0,6, а индексы — от −1 до 1, поэтому без масштабирования индексы определяли бы расстояния между пикселями сильнее каналов. Стандартизация приводит каждый признак к среднему 0 и стандартному отклонению 1: $z = (x - \mu) / \sigma$.

> 📚 **Подсказка:** лекция «Кластеризация спутниковых снимков в среде Google Earth Engine», **раздел 2.5** (масштабирование признаков); в GEE среднее и стандартное отклонение каналов по области даёт `image.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True), region, scale=..., bestEffort=True)`, ключи результата — `B2_mean`, `B2_stdDev` и т. д.

**Задание 5.1.** Допишите функцию `standardize(image, region)`: она рассчитывает по области средние и стандартные отклонения всех каналов (`scale=30` — для оценки статистик этого достаточно), возвращает стандартизированное изображение и два списка — средние и стандартные отклонения в порядке каналов. Получите `cluster_input, means, stds`.

In [ ]:
def standardize(image, region):
    """Стандартизация каналов изображения по статистикам области: (x - среднее) / стандартное отклонение."""
    names = image.bandNames().getInfo()
    stats = ...      # reduceRegion со средним и стандартным отклонением, scale=30, bestEffort=True, .getInfo()
    means = ...      # список средних в порядке names
    stds = ...       # список стандартных отклонений в порядке names
    standardized = ...   # (image - ee.Image.constant(means)) / ee.Image.constant(stds), каналы переименовать в names
    return standardized, means, stds

# Ваш код: cluster_input, means, stds = standardize(...) и таблица статистик

### **5.2. K-средних на серверах GEE при разных K**

> 📚 **Подсказка:** лекция «Кластеризация спутниковых снимков в среде Google Earth Engine», **раздел 3.5** (`ee.Clusterer.wekaKMeans`, `train`, `cluster`).

Число кластеров заранее неизвестно. При малом $K$ редкие объекты «прилипают» к крупным кластерам, при большом — крупные однородные поверхности дробятся на части. Поэтому построим разбиения сразу для нескольких $K$ и сравним их.

**Задание 5.2.** Получите обучающую выборку `training` из `cluster_input` (10 000 пикселей, `scale=SCALE`, `projection=utm_crs(roi)`, `seed=42`, `tileScale=4`). Для `K_LIST = [6, 12, 18]` обучите `ee.Clusterer.wekaKMeans(nClusters=K, init=1, seed=42)` и кластеризуйте `cluster_input`. Объедините карты в одно изображение `clusters` с каналами `K6`, `K12`, `K18` (`ee.Image.cat`), покажите кластеры при $K = 12$ на интерактивной карте (`randomVisualizer()`) и экспортируйте `clusters` в `data/clusters.tif` (тип `int16`). Загрузите результат в массив `cluster_maps` (каналы — разные $K$).

In [ ]:
K_LIST = [6, 12, 18]
# Ваш код: training, clusters (каналы K6, K12, K18), интерактивная карта, экспорт data/clusters.tif, cluster_maps

### **5.3. Площади и спектральные профили кластеров**

> 📚 **Подсказка:** лекция «Кластеризация спутниковых снимков в среде Google Earth Engine», **разделы 3.3–3.4** (спектральные профили, состав кластеров по Dynamic World).

Кроме профилей, оценим **необычность** каждого кластера — расстояние от его центроида до «среднего» пикселя территории в пространстве стандартизированных признаков. После стандартизации средний пиксель находится в начале координат, поэтому необычность — это просто длина вектора центроида: $\lVert c_k
Vert = \sqrt{\sum_j c_{kj}^2}$. Кластеры фоновых поверхностей, занимающих большую часть территории, лежат близко к среднему, а спектрально непохожие на фон группы — далеко от него, независимо от их площади. Центроиды кластеров GEE вычислим сами: это средние стандартизированные признаки пикселей каждого кластера.

**Задание 5.3.** Для разбиения с $K = 12$ (`labels = cluster_maps[1]`):
- стандартизируйте признаки пикселей `pixels` теми же `means` и `stds`, что и в GEE (массив `z`), и рассчитайте центроиды кластеров `centroids` (форма $(K, 8)$);
- постройте таблицу `profiles`: для каждого кластера — площадь (га), доля площади (%), необычность и средние значения восьми признаков `LOCAL_BANDS` (по массиву `bands`); отсортируйте её по площади;
- покажите карту кластеров рядом со снимком и график спектральных профилей шести каналов (по оси X — каналы, по оси Y — среднее отражение, по линии на кластер).

In [ ]:
K = 12
labels = cluster_maps[K_LIST.index(K)]
cluster_colors = plt.cm.tab20(np.linspace(0, 1, 20))[:K]
cluster_cmap = ListedColormap(cluster_colors)

In [ ]:
# Ваш код

### **5.4. Состав кластеров по Dynamic World и по своей классификации**

**Задание 5.4.** Постройте две таблицы сопряжённости, нормированные по строкам (%): кластер × класс Dynamic World (`dw`) и кластер × класс вашей классификации (`classification`). Строки упорядочите так же, как в `profiles` (по площади). Выведите их рядом в виде тепловых карт.

In [ ]:
# Ваш код

**Задание 5.4 (продолжение).** По профилям, составу и виду на карте дайте названия всем кластерам и заполните таблицу. Названия — ваша интерпретация, а не результат алгоритма.

#### **Таблица 3. Интерпретация кластеров (K = 12)**

| Кластер | Площадь, га | Спектральный профиль | Состав по Dynamic World | Название |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |

**Таблица 3:** _______________

### **5.5. Необычные кластеры: «иголка в стоге сена»**

Первые кандидаты на интересные объекты — кластеры, далёкие от фона: алгоритм отвёл им собственный центроид, потому что их спектр не похож на спектр основной части территории. Часто это и самые маленькие кластеры, но не всегда: крупный карьер или большая гарь могут занимать сотни гектаров и всё же резко отличаться от окружающего леса, а несколько маленьких кластеров могут оказаться оттенками фона. Поэтому ориентируемся на необычность, а площадь учитываем при интерпретации.

In [ ]:
#@title Готовая функция: фрагменты снимка с подсветкой пикселей маски
def zoom_views(mask, n=3, size=60, color=(1, 0, 0), title=''):
    """n фрагментов с наибольшей плотностью пикселей маски: снимок, B12-B8-B4 и подсветка."""
    step = 2 * size
    density = np.add.reduceat(np.add.reduceat(mask.astype(int), np.arange(0, H, step), axis=0), np.arange(0, W, step), axis=1)
    best = np.dstack(np.unravel_index(np.argsort(density.ravel())[::-1][:n], density.shape))[0]
    fig, axes = plt.subplots(3, n, figsize=(6 * n, 18))
    for j, (r, c) in enumerate(best):
        r = min(max(r * step + size, size), H - size)
        c = min(max(c * step + size, size), W - size)
        window = np.s_[r - size:r + size, c - size:c + size]
        highlight = rgb[window].copy()
        highlight[mask[window]] = color
        for i, img in enumerate([rgb[window], swir[window], highlight]):
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
        axes[0, j].set_title(f'Фрагмент {j + 1}: строка {r}, столбец {c}')
    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

**Задание 5.5.** Выведите три кластера с наибольшей необычностью с площадью и двумя главными классами Dynamic World и покажите для каждого карту расположения пикселей на снимке и фрагменты крупным планом (`zoom_views(labels == k, title=...)`). Затем выберите кластеры, которые вы считаете «иголками» — объектами, не имеющими своего класса в Dynamic World, — и запишите их номера в список `NEEDLES`.

In [ ]:
# Ваш код

In [ ]:
NEEDLES = [...]   # номера кластеров-«иголок»
needle_mask = np.isin(labels, NEEDLES) & valid
# Ваш код: площадь «иголок» и их состав по Dynamic World и по своей классификации, %

**Вопрос:** Что представляют собой выбранные кластеры? Опишите, как они выглядят на снимке в естественных цветах и в комбинации B12–B8–B4, их спектральный профиль, площадь и расположение. К каким классам их отнесли Dynamic World и ваша модель, и почему ни одна из карт не выделяет их отдельно?

**Ответ:** _______________

### **5.6. Как число кластеров меняет найденное**

**Задание 5.6.** Для каждого $K$ из `K_LIST` выведите три самых необычных кластера: необычность, площадь, доля площади, доля пикселей кластера, входящих в ваши «иголки» при $K = 12$ (`needle_mask`), и доля всех «иголок», попавших в этот кластер. Так видно, при каком $K$ интересующие вас объекты выделяются в отдельные кластеры и как они дробятся при большом $K$.

In [ ]:
# Ваш код

**Вопрос:** При каком наименьшем $K$ ваши объекты выделяются в собственный кластер? Что происходит с ними при $K = 6$ и при $K = 18$? Какое $K$ вы бы выбрали для поиска объектов этого типа на соседних территориях?

**Ответ:** _______________

### **5.7. Необычные пиксели вне кластеров**

Отдельные необычные объекты не образуют своего кластера, но их пиксели находятся далеко от всех центроидов (см. лекция «Кластеризация спутниковых снимков в среде Google Earth Engine», **раздел 5.3**). Центроиды `centroids` уже рассчитаны в разделе 5.3.

**Задание 5.7.** Рассчитайте расстояние от каждого пикселя до ближайшего центроида и карту `distance_map`. Найдите 0,1% самых далёких пикселей среди пикселей, не входящих в `NEEDLES`, покажите их на снимке и крупным планом. Определите, что это за объекты.

In [ ]:
# Совет: квадраты расстояний всех пикселей до всех центроидов без больших промежуточных массивов:
# (z ** 2).sum(axis=1)[:, None] - 2 * z @ centroids.T + (centroids ** 2).sum(axis=1)[None]
# Ваш код: расстояние до ближайшего центроида, distance_map, 0,1% самых далёких вне needle_mask, карта и zoom_views

**Вопрос:** Что оказалось среди самых необычных пикселей? Это отдельные объекты, границы между разными поверхностями или артефакты композита (остатки облаков, теней)? Есть ли среди них объекты, которые вы пропустили бы при просмотре снимка?

**Ответ:** _______________

### **5.8. Проверка гипотез и выводы по разделу 5**

**Задание 5.8.** Вернитесь к таблице 2. Для каждой гипотезы укажите, подтвердилась ли она, каким кластером или каким способом найден объект, его площадь и класс Dynamic World.

**Ответ:** _______________

Опишите, что нашла кластеризация из того, чего нет в легенде Dynamic World и что не выделил ваш классификатор; какие объекты плохо видны в естественных цветах, но выделяются по инфракрасным каналам; как выбор $K$ влияет на результат.

**Вывод:** _______________

---

## **6. Сжатие изображений кластеризацией цветов (для бакалавров)**

Кластеризация цветов заменяет все цвета изображения палитрой из $K$ цветов — центроидов кластеров (см. лекция «Кластеризация спутниковых снимков в среде Google Earth Engine», **раздел 6**). Выигрыш в размере получается, только если результат сохранён как **палитра + номера цветов пикселей**. В этом разделе вы проверите, для каких изображений и при каком числе цветов такое сжатие действительно выгодно по сравнению с обычными форматами.

### **6.1. Изображения для экспериментов**

**Задание 6.1.** Загрузите в папку `photos` (панель «Файлы» слева, папка создаётся следующей ячейкой) **две свои фотографии** в формате JPEG:
- фотографию с плавными переходами цвета — небо, портрет, пейзаж, животное на однородном фоне;
- фотографию с множеством мелких деталей и ярких цветов — рынок, цветник, городская улица.

Пропишите имена файлов в словаре `PHOTOS`. Кроме фотографий в эксперименты войдут два изображения вашей территории: фрагмент карты кластеров $K = 12$ из раздела 5 (тематическая карта с небольшим числом цветов) и тот же фрагмент снимка в естественных цветах. Все изображения уменьшаются так, чтобы большая сторона была не больше 1024 пикселей, — это ускоряет расчёты и делает изображения сопоставимыми.

In [ ]:
os.makedirs('photos', exist_ok=True)

In [ ]:
PHOTOS = {
    'Фото с плавными переходами': 'photos/....jpg',   # имя вашего файла
    'Фото с мелкими деталями': 'photos/....jpg',
}

In [ ]:
#@title Готовая функция: подготовка изображений
MAX_SIDE = 1024

def load_photo(path, max_side=MAX_SIDE):
    """RGB-массив фотографии, уменьшенной до max_side пикселей по большей стороне."""
    img = Image.open(path).convert('RGB')
    img.thumbnail((max_side, max_side))
    return np.array(img)

images = {name: load_photo(path) for name, path in PHOTOS.items()}

# Фрагмент территории: центральный квадрат растра не больше MAX_SIDE пикселей
side = min(H, W, MAX_SIDE)
r0, c0 = (H - side) // 2, (W - side) // 2
window = np.s_[r0:r0 + side, c0:c0 + side]
cluster_rgb = np.round(cluster_colors[:, :3] * 255).astype('uint8')   # цвета кластеров в RGB
images['Карта кластеров'] = cluster_rgb[np.clip(labels[window], 0, K - 1)]
images['Снимок Sentinel-2'] = (rgb[window] * 255).astype('uint8')

fig, axes = plt.subplots(1, len(images), figsize=(7 * len(images), 7))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(f'{name}, {img.shape[1]}×{img.shape[0]}, цветов: {len(np.unique(img.reshape(-1, 3), axis=0)):,}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### **6.2. Как сохранять результат**

Размер файла — главный показатель эффективности сжатия, поэтому сохраняйте изображения **строго одним из способов из таблицы 4** и измеряйте размер **настоящего файла на диске** функцией `os.path.getsize`. Все файлы раздела сохраняйте в папку `compressed/<название изображения>/`.

#### **Таблица 4. Способы сохранения**

| **Способ** | **Код** | **Что записывается в файл** |
|---|---|---|
| PNG без потерь — эталон размера | `Image.fromarray(img).save(path, format='PNG', optimize=True)` | 3 байта цвета на пиксель, сжатие без потерь |
| JPEG, качество 90 — обычное сжатие фотографий | `Image.fromarray(img).save(path, format='JPEG', quality=90)` | Сжатие с потерями, рассчитанное на фотографии |
| **Палитровый PNG — результат кластеризации** | `p_img = Image.frombytes('P', (w, h), indices.astype('uint8').tobytes())`<br>`p_img.putpalette(palette.flatten().tolist())`<br>`p_img.save(path, format='PNG', optimize=True)` | Палитра из $K$ цветов и номер цвета для каждого пикселя: 1 бит на пиксель при 2 цветах, 4 бита при 16, 8 бит при 256 |

Обратите внимание:
- номера цветов `indices` должны иметь тип `uint8` (от 0 до 255), поэтому $K \le 256$;
- параметр `mode` функции `Image.fromarray` в новых версиях Pillow устарел, поэтому палитровое изображение создаётся через `Image.frombytes`;
- изображение `palette[indices]` (восстановленные цвета) нужно только для просмотра и расчёта качества — если сохранить его как RGB-PNG или JPEG, выигрыша от кластеризации не будет (это вы проверите в разделе 6.4).

**Задание 6.2.** Допишите две функции:
- `quantize_kmeans(image, k, sample_size=50000, seed=42)` — обучает `KMeans(n_clusters=k, n_init=1, random_state=seed)` на случайной подвыборке `sample_size` пикселей (все пиксели, если `sample_size=None` или пикселей меньше), относит все пиксели к ближайшим центроидам и возвращает палитру `palette` (`uint8`, форма $(k, 3)$), индексы `indices` (`uint8`, форма $(h, w)$) и время работы в секундах;
- `save_palette_png(palette, indices, path)` — сохраняет палитровый PNG по таблице 4 и возвращает размер файла в КБ.

In [ ]:
def quantize_kmeans(image, k, sample_size=50000, seed=42):
    """Квантизация RGB-изображения до k цветов методом K-средних: палитра, индексы и время, с."""
    start = time.perf_counter()
    h_img, w_img, _ = image.shape
    colors = ...     # пиксели как массив (h*w, 3) типа float32
    sample = ...     # случайная подвыборка sample_size строк colors (или все строки)
    kmeans = ...     # KMeans(n_clusters=k, n_init=1, random_state=seed), обученный на sample
    palette = ...    # центроиды, округлённые до целых, тип uint8
    indices = ...    # номера кластеров всех пикселей, тип uint8, форма (h_img, w_img)
    return palette, indices, time.perf_counter() - start

def save_palette_png(palette, indices, path):
    """Палитровый PNG: таблица цветов и номер цвета каждого пикселя. Возвращает размер файла, КБ."""
    ...              # сохранение по таблице 4
    return os.path.getsize(path) / 1024

In [ ]:
#@title Готовые функции: сохранение без кластеризации и метрики качества
def save_rgb(img, path, **save_args):
    """Сохраняет RGB-массив в файл и возвращает размер, КБ."""
    Image.fromarray(img).save(path, **save_args)
    return os.path.getsize(path) / 1024

def quality(original, restored):
    """PSNR (дБ) и SSIM восстановленного изображения относительно исходного."""
    return (peak_signal_noise_ratio(original, restored, data_range=255),
            structural_similarity(original, restored, channel_axis=2, data_range=255))

for name in images:
    os.makedirs(f'compressed/{name}', exist_ok=True)

### **6.3. Размер, качество и время при разном числе цветов**

**Задание 6.3.** Для каждого изображения из `images`:
1. Сохраните эталоны без кластеризации: `compressed/<имя>/original.png` (PNG без потерь) и `compressed/<имя>/original_q90.jpg` (JPEG, качество 90). Для JPEG рассчитайте PSNR и SSIM относительно исходного изображения.
2. Для каждого `k` из `K_COLORS` выполните `quantize_kmeans`, сохраните палитровый PNG `compressed/<имя>/kmeans_<k>.png`, рассчитайте PSNR и SSIM восстановленного изображения `palette[indices]`.

Соберите две таблицы: `baselines` (столбцы `Изображение`, `PNG без потерь, КБ`, `JPEG 90, КБ`, `JPEG 90 PSNR`, `JPEG 90 SSIM`) и `results` (столбцы `Изображение`, `Цветов`, `Время, с`, `Палитровый PNG, КБ`, `PSNR`, `SSIM`). Сохраните палитры и индексы в словарь `quantized[(имя, k)]` — они понадобятся дальше. Если восстановленное изображение точно совпадает с исходным (так бывает, когда цветов в изображении не больше $K$), PSNR равен бесконечности.

In [ ]:
K_COLORS = [2, 4, 8, 16, 32, 64, 128, 256]

In [ ]:
# Ваш код

In [ ]:
#@title Готовый код: графики размера, качества и времени
fig, axes = plt.subplots(1, 4, figsize=(28, 6))
for name, part in results.groupby('Изображение', sort=False):
    ratio = part['Палитровый PNG, КБ'] / baselines.loc[name, 'PNG без потерь, КБ']
    line, = axes[0].plot(part['Цветов'], ratio, 'o-', linewidth=3, label=name)
    axes[0].axhline(baselines.loc[name, 'JPEG 90, КБ'] / baselines.loc[name, 'PNG без потерь, КБ'],
                    color=line.get_color(), linestyle='--', linewidth=2)
    axes[1].plot(part['Цветов'], part['PSNR'], 'o-', linewidth=3, color=line.get_color())
    axes[2].plot(part['Цветов'], part['SSIM'], 'o-', linewidth=3, color=line.get_color())
    axes[3].plot(part['Цветов'], part['Время, с'], 'o-', linewidth=3, color=line.get_color())
titles = ['Размер палитрового PNG / PNG без потерь\n(пунктир — JPEG 90)', 'PSNR, дБ', 'SSIM', 'Время квантизации, с']
for ax, title in zip(axes, titles):
    ax.set_xscale('log', base=2)
    ax.set_xticks(K_COLORS, K_COLORS)
    ax.set_xlabel('Число цветов K')
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].set_yscale('log')
axes[0].legend()
plt.tight_layout()
plt.show()
display(results.pivot(index='Цветов', columns='Изображение', values='Время, с').round(2))
results.pivot(index='Цветов', columns='Изображение', values='Палитровый PNG, КБ').round(1)

**Вопрос:** Во сколько раз палитровый PNG при 16 цветах меньше PNG без потерь для каждого изображения? Для какого изображения выигрыш наибольший и почему? Как растёт время квантизации с ростом $K$?

**Ответ:** _______________

### **6.4. Что будет, если сохранить результат неправильно**

**Задание 6.4.** Для каждого изображения при $K = 16$ сохраните восстановленное изображение `palette[indices]` как RGB-PNG (`kmeans_16_rgb.png`) и как JPEG качества 90 (`kmeans_16_q90.jpg`). Сравните размеры с палитровым PNG `kmeans_16.png` и с эталонами в таблице `wrong`.

In [ ]:
# Ваш код

**Вопрос:** Во сколько раз RGB-PNG и JPEG квантизованного изображения тяжелее палитрового PNG? Почему JPEG квантизованного изображения может оказаться не меньше JPEG исходного?

**Ответ:** _______________

### **6.5. Подвыборка пикселей и время**

K-средних можно обучать не на всех пикселях, а на случайной подвыборке: центроиды получаются почти такими же, а время сокращается. Проверим, где граница.

**Задание 6.5.** Для первой фотографии и $K = 64$ выполните `quantize_kmeans` с `sample_size` из списка `[1000, 10000, 50000, None]` (`None` — все пиксели). Постройте таблицу `sampling`: размер подвыборки, время, PSNR, SSIM.

In [ ]:
# Ваш код

**Вопрос:** Во сколько раз обучение на всех пикселях дольше, чем на 50 000? Насколько при этом меняется PSNR? Какой размер подвыборки вы бы рекомендовали и почему?

**Ответ:** _______________

### **6.6. K-средних против встроенной квантизации PIL**

В библиотеке PIL есть собственная быстрая квантизация методом **медианного сечения** (`Image.quantize`): цветовое пространство рекурсивно делится пополам по самой «длинной» оси, пока не получится $K$ ячеек. В отличие от K-средних, метод не итеративный и не минимизирует ошибку восстановления цветов явно.

**Задание 6.6.** Для всех изображений и $K \in \{4, 16, 64\}$ получите палитровое изображение `Image.fromarray(img).quantize(colors=k, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE)`, измерьте время, сохраните его как `mediancut_<k>.png` (`optimize=True`), восстановите цвета (`.convert('RGB')`) и рассчитайте PSNR и SSIM. Сравните с K-средних из таблицы `results` в таблице `versus`.

In [ ]:
# Ваш код

**Вопрос:** Какой метод даёт более высокое качество при одинаковом числе цветов и на сколько децибел? Для каких изображений разница больше всего? Как различаются размер файлов и время работы методов?

**Ответ:** _______________

### **6.7. Выбор числа цветов**

Выберем для каждого изображения наименьшее число цветов, при котором искажения малозаметны, и сравним размер файла с обычными форматами при этом качестве.

**Задание 6.7.** Для каждого изображения найдите наименьшее `k` из `K_COLORS`, при котором `SSIM` ≥ 0,95, и постройте таблицу `choice`: выбранное `k`, размер палитрового PNG, PSNR и SSIM при нём, размер PNG без потерь, размер, PSNR и SSIM JPEG 90, и отношение размера палитрового PNG к JPEG 90.

In [ ]:
# Ваш код

In [ ]:
#@title Готовый код: исходные изображения и выбранное число цветов
fig, axes = plt.subplots(2, len(images), figsize=(7 * len(images), 14))
for j, (name, img) in enumerate(images.items()):
    k = int(choice.loc[name, 'K'])
    palette, indices = quantized[(name, k)]
    axes[0, j].imshow(img)
    axes[0, j].set_title(f'{name}: исходное')
    axes[1, j].imshow(palette[indices])
    axes[1, j].set_title(f'K = {k}: {choice.loc[name, "Палитровый PNG, КБ"]:.0f} КБ, SSIM {choice.loc[name, "SSIM"]:.3f}')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

**Вопрос:** Для каких изображений палитровый PNG при малозаметных искажениях меньше JPEG 90, а для каких — больше? Чем отличаются эти изображения? Какой способ хранения вы выберете для фотографий, для тематических карт и для снимков в естественных цветах?

**Ответ:** _______________

### **6.8. Выводы по разделу 6**

Опишите, при каких условиях кластеризация цветов уменьшает размер файла, как правильно сохранять результат, как связаны число цветов, размер и качество, и чем K-средних отличается от встроенной квантизации.

**Вывод:** _______________

---

## **7. Итоговые выводы**

В выводах обязательно отразить:
1. **Территория и найденные объекты**: что нашла кластеризация, их площадь, вид в естественных цветах и в инфракрасных каналах;
2. **Классификация по Dynamic World**: точность при случайной и пространственной проверке, лучший классификатор, проблемные классы, совпадение карты с Dynamic World;
3. **Классификация и кластеризация**: что может и чего не может каждый подход на вашей территории;
4. **Сжатие изображений**: для каких изображений и при каком числе цветов кластеризация выгодна, во сколько раз уменьшается размер;
5. **Возможное практическое применение** полученных результатов.

**Выводы:** _______________

---

## **Что сдавать**

1. Выполненный блокнот со всеми выводами ячеек и заполненными ответами (таблицы 2 и 3, ответы на вопросы, выводы по разделам 4–6 и итоговые выводы).
2. Папку `compressed` с файлами раздела 6 (в архиве).

## **Дополнительные задания (выполняются по желанию)**

1. **Перенос модели.** Примените классификатор, обученный на своей территории, к композиту другой территории из таблицы 1 и рассчитайте совпадение с Dynamic World. Сравните с точностью пространственной проверки и объясните разницу.
2. **Поиск по эталону.** Возьмите центроиды «иголок» своей территории и найдите на другой территории пиксели, ближе всего к ним расположенные в пространстве стандартизированных признаков. Нашлись ли объекты того же типа?
3. **Собственная реализация K-средних.** Напишите K-средних для цветов на NumPy (инициализация, назначение пикселей, пересчёт центроидов, остановка по сдвигу центроидов) и сравните с `KMeans` по времени, числу итераций и PSNR при $K = 16$.
4. **Другое цветовое пространство.** Выполните квантизацию фотографий в пространстве CIELAB (`skimage.color.rgb2lab`), где евклидово расстояние ближе к различию цветов, воспринимаемому глазом, и сравните SSIM с квантизацией в RGB при одинаковом $K$.